#### Initialize the libraries, the environment and the client

In [1]:
import os
from pprint import pprint

import pandas as pd

try:
    from epo.tipdata.ops import OPSClient, exceptions, models
except ImportError:
    os.system("pip install epo-tipdata-ops")
    from epo.tipdata.ops import OPSClient, exceptions, models

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

client = OPSClient(key=os.getenv("OPS_KEY"), secret=os.getenv("OPS_SECRET"))

### 7.7 Register_search Method
#### 7.7.1 register_search(cql, range_begin, range_end, output_type)
* Enables you to search for specific register data associated with patents within the public aspect of the patent lifecycle.<br>
* Parameters:<br>
```cql (str)```: CQL search query string<br>
```range_begin (int, optional)```: Starting position of the results (defaults to 1)<br>
```range_end (int, optional)```: Maximum number of results to retrieve (defaults to 25)<br>
```output_type (str)```: Format of the returned data (raw XML or Pandas DataFrame)<br>

#### 7.7.2 Metadata and Search Result Output from EPO OPS (Open Patent Services) API Register Search
This example demonstrates searching for European patents from SIEMENS with unitary effect using a CQL query and retrieving specific register data, like ufd, that are part of the patent lifecycle, providing once a XML output read with pprint() and once a Pandas dataframe.

Examples of C0 publications: EP3785601 C0 (EP20186735), EP3606080 C0 (EP18186812), EP3701530 C0 (EP18871135)

In [2]:
import json

import pandas as pd

# Ensure long text values are fully visible
pd.set_option("display.max_colwidth", 2000)

# Retrieve data from OPS
content = client.register_search(
    cql="pa=SIEMENS and pn=epb and ufd=yes",
    range_begin=1,
    range_end=100,
    output_type="dataframe",
)

# Display retrieved data
display(content)

# Define column names
columns_of_interest = [
    "ops:world-patent-data|ops:register-search|total-result-count",
    "ops:world-patent-data|ops:register-search|ops:query",
    "ops:world-patent-data|ops:register-search|ops:range",
    "ops:world-patent-data|ops:register-search|reg:register-documents",
]

# Ensure all required columns exist
missing_columns = [col for col in columns_of_interest if col not in content.columns]
if missing_columns:
    print(f"Warning: Missing columns - {missing_columns}")

# Convert JSON-like strings to dictionaries (if applicable)
for col in columns_of_interest:
    if col in content.columns:
        content[col] = content[col].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )

# Extract first row (assuming it represents a single dataset)
if not content.empty:
    extracted_data = {
        col: content[col].iloc[0]
        for col in columns_of_interest
        if col in content.columns
    }
else:
    extracted_data = {}

# Ensure total-result-count remains as an integer and does not get expanded
if isinstance(
    extracted_data.get("ops:world-patent-data|ops:register-search|total-result-count"),
    (int, float),
):
    extracted_data["ops:world-patent-data|ops:register-search|total-result-count"] = (
        extracted_data["ops:world-patent-data|ops:register-search|total-result-count"]
    )

# Debug: Print extracted data structure
print("\nExtracted Data:")
for key, value in extracted_data.items():
    print(f"{key}: {type(value)}")

# **No Flattening Here**: Keep as a structured DataFrame instead of `pd.json_normalize`
structured_output = pd.DataFrame([extracted_data])

# Display available columns
print("\nFinal DataFrame Columns:", structured_output.columns.tolist())

# Display final structured output
display(structured_output)

,ops:world-patent-data|xmlns:ops,ops:world-patent-data|xmlns:reg,ops:world-patent-data|xmlns:xlink,ops:world-patent-data|xmlns:cpc,ops:world-patent-data|xmlns:cpcdef,ops:world-patent-data|ops:register-search|total-result-count,ops:world-patent-data|ops:register-search|ops:query,ops:world-patent-data|ops:register-search|ops:range,ops:world-patent-data|ops:register-search|reg:register-documents
0,http://ops.epo.org,http://www.epo.org/register,http://www.w3.org/1999/xlink,http://www.epo.org/cpcexport,http://www.epo.org/cpcdefinition,1764,"{'@syntax': 'CQL', '#text': 'pa=SIEMENS and pn=epb and ufd=yes'}","{'@begin': '1', '@end': '100'}","{'@produced-by': 'RO', 'reg:register-document': [{'@produced-by': 'RO', 'reg:ep-patent-statuses': {'reg:ep-patent-status': {'@change-date': '', '@status-code': ''}}, 'reg:bibliographic-data': {'reg:publication-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '4611493', 'reg:date': '20250903'}}, 'reg:classifications-ipcr': {'reg:classification-ipcr': {'@sequence': 'CLASSIFICATION_1', 'reg:text': 'H05K3/12;; B41F33/00;; B41F15/44;; B41F15/08;; B41F15/36'}}, 'reg:application-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '24160500'}}, 'reg:parties': {'reg:applicants': {'reg:applicant': {'@app-type': 'applicant', '@designation': 'all', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Siemens Aktiengesellschaft', 'reg:address': {'reg:country': None}}, 'reg:nationality': {'reg:country': None}, 'reg:residence': {'reg:country': None}}}, 'reg:agents': {'reg:agent': {'@rep-type': 'agent', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Siemens Patent Attorneys', 'reg:address': {'reg:country': None}}}}}, 'reg:invention-title': [{'@lang': 'de', '#text': 'VERFAHREN ZUR ÜBERWACHUNG DER DRUCKBEREITSCHAFT EINER DRUCKMASKE, COMPUTERPROGRAMM UND COMPUTERSYSTEM'}, {'@lang': 'en', '#text': 'METHOD FOR MONITORING THE PRINTING READINESS OF A PRINTING MASK, COMPUTER PROGRAM AND COMPUTER SYSTEM'}, {'@lang': 'fr', '#text': ""PROCÉDÉ DE SURVEILLANCE DE LA DISPONIBILITÉ DE PRESSION D'UN MASQUE D'IMPRESSION, PROGRAMME INFORMATIQUE ET SYSTÈME INFORMATIQUE""}]}, 'reg:unitary-patent': {'@date-of-request': '20260217'}}, {'@produced-by': 'RO', 'reg:ep-patent-statuses': {'reg:ep-patent-status': {'@change-date': '', '@status-code': ''}}, 'reg:bibliographic-data': {'reg:publication-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '4529853', 'reg:date': '20250402'}}, 'reg:classifications-ipcr': {'reg:classification-ipcr': {'@sequence': 'CLASSIFICATION_1', 'reg:text': 'A61B6/42;; G01T1/00'}}, 'reg:application-reference': {'reg..."



Extracted Data:
ops:world-patent-data|ops:register-search|total-result-count: <class 'numpy.int64'>
ops:world-patent-data|ops:register-search|ops:query: <class 'dict'>
ops:world-patent-data|ops:register-search|ops:range: <class 'dict'>
ops:world-patent-data|ops:register-search|reg:register-documents: <class 'dict'>

Final DataFrame Columns: ['ops:world-patent-data|ops:register-search|total-result-count', 'ops:world-patent-data|ops:register-search|ops:query', 'ops:world-patent-data|ops:register-search|ops:range', 'ops:world-patent-data|ops:register-search|reg:register-documents']


,ops:world-patent-data|ops:register-search|total-result-count,ops:world-patent-data|ops:register-search|ops:query,ops:world-patent-data|ops:register-search|ops:range,ops:world-patent-data|ops:register-search|reg:register-documents
0,1764,"{'@syntax': 'CQL', '#text': 'pa=SIEMENS and pn=epb and ufd=yes'}","{'@begin': '1', '@end': '100'}","{'@produced-by': 'RO', 'reg:register-document': [{'@produced-by': 'RO', 'reg:ep-patent-statuses': {'reg:ep-patent-status': {'@change-date': '', '@status-code': ''}}, 'reg:bibliographic-data': {'reg:publication-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '4611493', 'reg:date': '20250903'}}, 'reg:classifications-ipcr': {'reg:classification-ipcr': {'@sequence': 'CLASSIFICATION_1', 'reg:text': 'H05K3/12;; B41F33/00;; B41F15/44;; B41F15/08;; B41F15/36'}}, 'reg:application-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '24160500'}}, 'reg:parties': {'reg:applicants': {'reg:applicant': {'@app-type': 'applicant', '@designation': 'all', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Siemens Aktiengesellschaft', 'reg:address': {'reg:country': None}}, 'reg:nationality': {'reg:country': None}, 'reg:residence': {'reg:country': None}}}, 'reg:agents': {'reg:agent': {'@rep-type': 'agent', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Siemens Patent Attorneys', 'reg:address': {'reg:country': None}}}}}, 'reg:invention-title': [{'@lang': 'de', '#text': 'VERFAHREN ZUR ÜBERWACHUNG DER DRUCKBEREITSCHAFT EINER DRUCKMASKE, COMPUTERPROGRAMM UND COMPUTERSYSTEM'}, {'@lang': 'en', '#text': 'METHOD FOR MONITORING THE PRINTING READINESS OF A PRINTING MASK, COMPUTER PROGRAM AND COMPUTER SYSTEM'}, {'@lang': 'fr', '#text': ""PROCÉDÉ DE SURVEILLANCE DE LA DISPONIBILITÉ DE PRESSION D'UN MASQUE D'IMPRESSION, PROGRAMME INFORMATIQUE ET SYSTÈME INFORMATIQUE""}]}, 'reg:unitary-patent': {'@date-of-request': '20260217'}}, {'@produced-by': 'RO', 'reg:ep-patent-statuses': {'reg:ep-patent-status': {'@change-date': '', '@status-code': ''}}, 'reg:bibliographic-data': {'reg:publication-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '4529853', 'reg:date': '20250402'}}, 'reg:classifications-ipcr': {'reg:classification-ipcr': {'@sequence': 'CLASSIFICATION_1', 'reg:text': 'A61B6/42;; G01T1/00'}}, 'reg:application-reference': {'reg..."


#### Explanation
This code registers a search task using the OPSClient class.<br>
##### 1. We define Search Parameters:<br>
```cql='pa = SIEMENS and ufd = yes'```: This line defines the search query in the client-specific query language (CQL). Here, "pa" indicates searching a field named "patent applicant" and "SIEMENS" is the value to search for (documents where the applicant is SIEMENS), while "ufd = yes" indicates the presence of a unitary effect request date .<br>
```range_begin=1```: This sets the starting document number for the search (1 in this case).<br>
```range_end=10```: This sets the ending document number for the search (10 in this case).<br>
```output_type='raw'```: This specifies the desired output format as raw data (likely bytes or text depending on the system). the parameter ```Dataframe``` can be used instead<br>

##### 2. We register a Search Task:<br>
```content = client.register_search(...)```: This line calls a method specific to the client library named register_search. This method is used to submit a search request with the defined parameters. The response from the server is stored in the content variable.<br>

##### 3. Key Difference from prior method:<br>
This code registers a search task and doesn't retrieve the results immediately. The server processes the search asynchronously and provide the results later through a different method call.
The prior method (doc_start=1 and the one retrieving PDF) perform the search and retrieve the results within the same code block.<br>

In [3]:
import json


# Function to return pretty-printed nested data as a string
def pretty_print(data, indent=0):
    """Recursively returns JSON-like nested structures as a string with indentation."""
    result = ""
    if isinstance(data, dict):
        for key, value in data.items():
            result += "  " * indent + str(key) + ":\n"
            result += pretty_print(value, indent + 1)
    elif isinstance(data, list):
        for i, item in enumerate(data):
            result += "  " * indent + f"Item {i}:\n"
            result += pretty_print(item, indent + 1)
    else:
        result += "  " * indent + str(data) + "\n"
    return result


# Define all column names of interest
columns_of_interest = [
    "ops:world-patent-data|ops:register-search|total-result-count",
    "ops:world-patent-data|ops:register-search|ops:query",
    "ops:world-patent-data|ops:register-search|ops:range",
    "ops:world-patent-data|ops:register-search|reg:register-documents",
]

# Initialize a variable to store the output
pretty_printed_data = ""

# Ensure content is not empty before proceeding
if content.empty:
    print("Error: The 'content' DataFrame is empty.")
else:
    # Iterate through each column and pretty-print its content
    for col in columns_of_interest:
        if col in content.columns:
            nested_data = content[col].iloc[0]  # Assuming one row; adjust as needed
            pretty_printed_data += f"\n🔹 **Extracted Data from Column:** {col}\n"
            pretty_printed_data += pretty_print(
                nested_data
            )  # Capture the pretty print result
        else:
            pretty_printed_data += (
                f"\n⚠️ **Warning:** Column '{col}' not found in content."
            )

print("pretty_printed_data:", pretty_printed_data)

pretty_printed_data: 
🔹 **Extracted Data from Column:** ops:world-patent-data|ops:register-search|total-result-count
1764

🔹 **Extracted Data from Column:** ops:world-patent-data|ops:register-search|ops:query
@syntax:
  CQL
#text:
  pa=SIEMENS and pn=epb and ufd=yes

🔹 **Extracted Data from Column:** ops:world-patent-data|ops:register-search|ops:range
@begin:
  1
@end:
  100

🔹 **Extracted Data from Column:** ops:world-patent-data|ops:register-search|reg:register-documents
@produced-by:
  RO
reg:register-document:
  Item 0:
    @produced-by:
      RO
    reg:ep-patent-statuses:
      reg:ep-patent-status:
        @change-date:
          
        @status-code:
          
    reg:bibliographic-data:
      reg:publication-reference:
        reg:document-id:
          reg:country:
            EP
          reg:doc-number:
            4611493
          reg:date:
            20250903
      reg:classifications-ipcr:
        reg:classification-ipcr:
          @sequence:
            CLASSIFICATI

In [4]:
!pip install anytree
from anytree import Node, RenderTree


def build_tree(data, parent):
    """Recursively build anytree structure from nested dictionary data."""
    if isinstance(data, dict):
        for key, value in data.items():
            child = Node(
                f"{key}: {value if not isinstance(value, (dict, list)) else ''}",
                parent=parent,
            )
            build_tree(value, child)  # Recursively add children
    elif isinstance(data, list):
        for i, item in enumerate(data):
            child = Node(f"Item {i}", parent=parent)
            build_tree(item, child)
    else:
        Node(str(data), parent=parent)  # Leaf nodes (e.g., plain values)


# Define all column names of interest
columns_of_interest = [
    "ops:world-patent-data|ops:register-search|total-result-count",
    "ops:world-patent-data|ops:register-search|ops:query",
    "ops:world-patent-data|ops:register-search|ops:range",
    "ops:world-patent-data|ops:register-search|reg:register-documents",
]

# Ensure content is not empty before proceeding
if content.empty:
    print("Error: The 'content' DataFrame is empty.")
else:
    for col in columns_of_interest:
        if col in content.columns:
            nested_data = content[col].iloc[0]  # Assuming one row; adjust as needed
            print(f"\n🔹 **Tree Representation of Column:** {col}")

            # Create root node for the column
            root = Node(col)

            # Build tree from nested data
            build_tree(nested_data, root)

            # Render tree structure
            for pre, fill, node in RenderTree(root):
                print(f"{pre}{node.name}")
        else:
            print(f"\n⚠️ **Warning:** Column '{col}' not found in content.")


🔹 **Tree Representation of Column:** ops:world-patent-data|ops:register-search|total-result-count
ops:world-patent-data|ops:register-search|total-result-count
└── 1764

🔹 **Tree Representation of Column:** ops:world-patent-data|ops:register-search|ops:query
ops:world-patent-data|ops:register-search|ops:query
├── @syntax: CQL
│   └── CQL
└── #text: pa=SIEMENS and pn=epb and ufd=yes
    └── pa=SIEMENS and pn=epb and ufd=yes

🔹 **Tree Representation of Column:** ops:world-patent-data|ops:register-search|ops:range
ops:world-patent-data|ops:register-search|ops:range
├── @begin: 1
│   └── 1
└── @end: 100
    └── 100

🔹 **Tree Representation of Column:** ops:world-patent-data|ops:register-search|reg:register-documents
ops:world-patent-data|ops:register-search|reg:register-documents
├── @produced-by: RO
│   └── RO
└── reg:register-document: 
    ├── Item 0
    │   ├── @produced-by: RO
    │   │   └── RO
    │   ├── reg:ep-patent-statuses: 
    │   │   └── reg:ep-patent-status: 
    │   │     

In [5]:
# Ensure the column exists and is not empty
if (
    "ops:world-patent-data|ops:register-search|reg:register-documents"
    in content.columns
    and not content.empty
):

    # Initialize lists to store the extracted data
    produced_by = []
    status_code = []
    publication_country = []
    publication_doc_number = []
    publication_date = []
    classification_sequence = []
    classification_text = []
    applicant_name = []
    agent_name = []
    invention_title_lang_en = []

    # Process each register document entry
    for entry in content[
        "ops:world-patent-data|ops:register-search|reg:register-documents"
    ]:
        print(
            content[
                "ops:world-patent-data|ops:register-search|reg:register-documents"
            ].head()
        )
        if isinstance(entry, dict) and "reg:register-document" in entry:
            # Iterate over the list of register documents
            for document in entry["reg:register-document"]:
                produced_by.append(document.get("@produced-by", None))

                # Extract EP Patent Status
                ep_statuses = document.get("reg:ep-patent-statuses", {})
                ep_status = ep_statuses.get("reg:ep-patent-status", {})
                status_code.append(ep_status.get("@status-code", None))

                # Extract Publication Reference Data
                pub_ref = (
                    document.get("reg:bibliographic-data", {})
                    .get("reg:publication-reference", {})
                    .get("reg:document-id", {})
                )
                publication_country.append(pub_ref.get("reg:country", None))
                publication_doc_number.append(pub_ref.get("reg:doc-number", None))
                publication_date.append(pub_ref.get("reg:date", None))

                # Extract Classification IPCR Data
                classification_ipcr = (
                    document.get("reg:bibliographic-data", {})
                    .get("reg:classifications-ipcr", {})
                    .get("reg:classification-ipcr", {})
                )
                classification_sequence.append(
                    classification_ipcr.get("@sequence", None)
                )
                classification_text.append(classification_ipcr.get("reg:text", None))

                # Extract Applicant Name
                applicant = (
                    document.get("reg:bibliographic-data", {})
                    .get("reg:parties", {})
                    .get("reg:applicants", {})
                    .get("reg:applicant", {})
                )
                applicants = applicant if isinstance(applicant, list) else [applicant]
                _applicant_names = []
                for _applicant in applicants:
                    _applicant_names.append(
                        _applicant.get("reg:addressbook", {}).get("reg:name", None)
                    )
                applicant_name.append(
                    ", ".join(_applicant_names) if _applicant_names else None
                )

                # Extract Agent Name
                agent = (
                    document.get("reg:bibliographic-data", {})
                    .get("reg:parties", {})
                    .get("reg:agents", {})
                    .get("reg:agent", {})
                )
                agent_name.append(
                    agent.get("reg:addressbook", {}).get("reg:name", None)
                )

                # Extract English Invention Title
                invention_title = next(
                    (
                        title.get("#text")
                        for title in document.get("reg:bibliographic-data", {}).get(
                            "reg:invention-title", []
                        )
                        if title.get("@lang") == "en"
                    ),
                    None,
                )
                invention_title_lang_en.append(invention_title)
        else:
            print(
                "⚠️ Warning: Expected dictionary structure but found something else in 'register-documents'. Skipping..."
            )

    # Create DataFrame with columns based on extracted lists
    df = pd.DataFrame(
        {
            "Produced By": produced_by,
            "Status Code": status_code,
            "Publication Country": publication_country,
            "Publication Doc Number": publication_doc_number,
            "Publication Date": publication_date,
            "Classification Sequence": classification_sequence,
            "Classification Text": classification_text,
            "Applicant Name": applicant_name,
            "Agent Name": agent_name,
            "Invention Title (EN)": invention_title_lang_en,
        }
    )

    # Display the DataFrame
    display(df)
else:
    print(
        "⚠️ Warning: Column 'ops:world-patent-data|ops:register-search|reg:register-documents' is missing or empty in the DataFrame."
    )

0    {'@produced-by': 'RO', 'reg:register-document': [{'@produced-by': 'RO', 'reg:ep-patent-statuses': {'reg:ep-patent-status': {'@change-date': '', '@status-code': ''}}, 'reg:bibliographic-data': {'reg:publication-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '4611493', 'reg:date': '20250903'}}, 'reg:classifications-ipcr': {'reg:classification-ipcr': {'@sequence': 'CLASSIFICATION_1', 'reg:text': 'H05K3/12;; B41F33/00;; B41F15/44;; B41F15/08;; B41F15/36'}}, 'reg:application-reference': {'reg:document-id': {'reg:country': 'EP', 'reg:doc-number': '24160500'}}, 'reg:parties': {'reg:applicants': {'reg:applicant': {'@app-type': 'applicant', '@designation': 'all', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Siemens Aktiengesellschaft', 'reg:address': {'reg:country': None}}, 'reg:nationality': {'reg:country': None}, 'reg:residence': {'reg:country': None}}}, 'reg:agents': {'reg:agent': {'@rep-type': 'agent', '@sequence': '1', 'reg:addressbook': {'reg:name': 'Si

,Produced By,Status Code,Publication Country,Publication Doc Number,Publication Date,Classification Sequence,Classification Text,Applicant Name,Agent Name,Invention Title (EN)
0,RO,,EP,4611493,20250903,CLASSIFICATION_1,H05K3/12;; B41F33/00;; B41F15/44;; B41F15/08;; B41F15/36,Siemens Aktiengesellschaft,Siemens Patent Attorneys,"METHOD FOR MONITORING THE PRINTING READINESS OF A PRINTING MASK, COMPUTER PROGRAM AND COMPUTER SYSTEM"
1,RO,,EP,4529853,20250402,CLASSIFICATION_1,A61B6/42;; G01T1/00,Siemens Healthineers AG,Siemens Healthineers Patent Attorneys,IMAGING SYSTEM AND METHOD FOR CAPTURING PROJECTION IMAGES
2,RO,,EP,4527301,20250326,CLASSIFICATION_1,A61B6/03;; A61B6/00,Siemens Healthineers AG,Siemens Healthineers Patent Attorneys,COMPUTED TOMOGRAPHY APPARATUS WITH IMPROVED DATA TRANSMISSION
3,RO,,EP,4527300,20250326,CLASSIFICATION_1,A61B6/03;; A61B6/00,Siemens Healthineers AG,Siemens Healthineers Patent Attorneys,COMPUTED TOMOGRAPHY APPARATUS WITH IMPROVED DATA TRANSMISSION
4,RO,,EP,4529086,20250326,CLASSIFICATION_1,H04L41/00;; H04L12/12;; H04L43/0811;; H04L9/40,Siemens Aktiengesellschaft,Siemens Patent Attorneys,DEVICE FOR CRYPTOGRAPHIC PROTECTION OF A COMMUNICATION DEVICE FOR AN INDUSTRIAL AUTOMATION SYSTEM
...,...,...,...,...,...,...,...,...,...,...
95,RO,,EP,4321892,20240214,CLASSIFICATION_1,G01R33/561,Siemens Healthineers AG,Siemens Healthineers Patent Attorneys,"METHOD FOR SUBJECT-SPECIFIC OPTIMISATION OF A MULTI-BAND RF PULSE, COMPUTER PROGRAM, COMPUTER READABLE MEDIUM AND MAGNETIC RESONANCE SYSTEM AND CONTROL UNIT FOR CARRYING OUT THE METHOD"
96,RO,,EP,4321103,20240214,CLASSIFICATION_1,A61B6/12;; A61B90/00;; A61B6/00;; A61B6/50,Siemens Healthineers AG,Kraus & Lederer PartGmbB,DYNAMIC VESSEL ROADMAP GUIDANCE
97,RO,,EP,4321887,20240214,CLASSIFICATION_1,G01R33/34;; G01R33/3415;; G01R33/36;; G01R33/28,Siemens Healthineers AG,Siemens Healthineers Patent Attorneys,LOCAL COIL FOR MAGNETIC RESONANCE TOMOGRAPHY SYSTEM AND MAGNETIC RESONANCE TOMOGRAPHY SYSTEM
98,RO,,EP,4321099,20240214,CLASSIFICATION_1,A61B6/00;; A61B6/50;; A61B6/03;; A61B6/46,Siemens Healthineers AG,Kraus & Lederer PartGmbB,METHOD AND SYSTEM FOR IMPROVED VESSEL COVERAGE FROM ANGIO-GRAPHY
